# Polity assignment
How individuals are assigned to Cliopatria polities, by assignment method.

In [1]:
import tempfile

import duckdb
import polars as pl

DB_PATH = '../data/cultura/humans_clean_enriched_v3.duckdb'

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(60)

polars.config.Config

## Load data
### Individuals per assignment method

In [2]:
con = duckdb.connect(DB_PATH, read_only=True)
con.sql(f"SET memory_limit='4GB'; SET threads=6; SET temp_directory='{tempfile.mkdtemp()}';")

method_counts = con.sql("""
    SELECT   match.assignation_method   AS assignation_method,
             COUNT(DISTINCT entity.qid) AS n_individuals
    FROM     individual_enriched, unnest(polity) AS unnested(match)
    GROUP BY match.assignation_method
""").pl()

n_matched = con.sql("SELECT COUNT(*) FROM individual_enriched WHERE len(polity) > 0").fetchone()[0]
con.close()

assert n_matched == method_counts['n_individuals'].sum(), 'each individual is matched by a single method'
print(f'matched individuals: {n_matched:,}')
method_counts

matched individuals: 5,150,199


assignation_method,n_individuals
str,i64
"""polygon_of_country_of_citizenship""",444084
"""polygon_of_birthplace""",709684
"""url_of_country_of_citizenship""",3181006
"""url_of_birthplace""",427
"""polygon_of_deathplace""",814694
"""url_of_deathplace""",304


## Table: new individuals matched by each method
Methods are listed in cascade order: an individual is assigned by the first method that matches, so each row counts only new individuals.

In [3]:
CASCADE_ORDER = [
    ('polygon_of_deathplace',             'Polygon merging', 'Place of death'),
    ('polygon_of_birthplace',             'Polygon merging', 'Place of birth'),
    ('polygon_of_country_of_citizenship', 'Polygon merging', 'Country of citizenship'),
    ('url_of_country_of_citizenship',     'URL matching',    'Country of citizenship'),
    ('url_of_deathplace',                 'URL matching',    'Place of death'),
    ('url_of_birthplace',                 'URL matching',    'Place of birth'),
]

assignment_table = (
    pl.DataFrame(CASCADE_ORDER, schema=['assignation_method', 'method', 'origin'], orient='row')
    .with_row_index('step', offset=1)
    .join(method_counts, on='assignation_method', how='left')
    .with_columns(pl.col('n_individuals').fill_null(0))
    .with_columns((pl.col('n_individuals') / n_matched * 100).round(2).alias('pct_individuals'))
    .select('step', 'method', 'origin', 'n_individuals', 'pct_individuals')
)

assert assignment_table['n_individuals'].sum() == n_matched
assignment_table

step,method,origin,n_individuals,pct_individuals
u32,str,str,i64,f64
1,"""Polygon merging""","""Place of death""",814694,15.82
2,"""Polygon merging""","""Place of birth""",709684,13.78
3,"""Polygon merging""","""Country of citizenship""",444084,8.62
4,"""URL matching""","""Country of citizenship""",3181006,61.76
5,"""URL matching""","""Place of death""",304,0.01
6,"""URL matching""","""Place of birth""",427,0.01
